# params-iterable-vs-groups — faded example 3: nn.Parameter routes through the Tensor branch

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `params-iterable-vs-groups`. The last cell reports your progress on the `Config: params iterable vs groups` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: params iterable vs groups` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`params-iterable-vs-groups`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "params-iterable-vs-groups"
DD_SUBTOPIC = "Config: params iterable vs groups"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`nn.Parameter` is a subclass of `torch.Tensor`, so `isinstance(p, torch.Tensor)` returns `True` for it. This means the flat-iterable branch of the params dispatch correctly handles the common case of passing `model.parameters()` — a generator of `nn.Parameter` objects — without any special-case logic. Never use `type(x) is t.Tensor` for this check, because that would exclude `nn.Parameter`.

## Faded exercise 3

### Exercise — nn.Parameter routes through the Tensor branch

Complete `normalize_with_parameter(params, default_lr)`. The function normalizes a params iterable that may contain `nn.Parameter` objects. The key requirement: use `isinstance(first, t.Tensor)` (not `type(first) is t.Tensor`) so that `nn.Parameter` passes the check.

Fill in the dispatch condition — the isinstance check that accepts both raw Tensors and Parameters.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def normalize_with_parameter(params, default_lr):
    materialized = list(params)
    if not materialized:
        raise ValueError('optimizer got an empty parameter list')
    first = materialized[0]
    if None:  # TODO: fill in this step — read the prompt cell above
        return [{'params': materialized, 'lr': default_lr}]
    if isinstance(first, dict):
        out = []
        for g in materialized:
            gc = dict(g)
            if 'lr' not in gc:
                gc['lr'] = default_lr
            out.append(gc)
        return out
    raise TypeError(f'Unexpected type: {type(first).__name__}')

# Test
t.manual_seed(9)
net = t.nn.Linear(4, 2)
result = normalize_with_parameter(net.parameters(), default_lr=1e-3)
print(len(result), result[0]['lr'])


def _test():
    import torch as t
    t.manual_seed(9)
    net = t.nn.Linear(4, 2)
    params = list(net.parameters())
    # All params are nn.Parameter — subclass of Tensor
    assert all(isinstance(p, t.nn.Parameter) for p in params)
    result = normalize_with_parameter(params, default_lr=1e-3)
    assert len(result) == 1, 'flat list should become one group'
    assert result[0]['lr'] == 1e-3
    assert len(result[0]['params']) == len(params)
    # Verify nn.Parameter goes through tensor branch, not dict branch
    for p_orig, p_stored in zip(params, result[0]['params']):
        assert p_orig is p_stored


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def normalize_with_parameter(params, default_lr):
    materialized = list(params)
    if not materialized:
        raise ValueError('optimizer got an empty parameter list')
    first = materialized[0]
    if isinstance(first, t.Tensor):  # TODO: Write the isinstance condition that correctly dispatches nn.Parameter objects into the Tensor branch.
        return [{'params': materialized, 'lr': default_lr}]
    if isinstance(first, dict):
        out = []
        for g in materialized:
            gc = dict(g)
            if 'lr' not in gc:
                gc['lr'] = default_lr
            out.append(gc)
        return out
    raise TypeError(f'Unexpected type: {type(first).__name__}')

# Test
t.manual_seed(9)
net = t.nn.Linear(4, 2)
result = normalize_with_parameter(net.parameters(), default_lr=1e-3)
print(len(result), result[0]['lr'])
```
</details>